# Day 4: PHI Governance & De-Identification Engine

This interactive demo compares raw vs. masked synthetic patient dataframes using Format-Preserving Encryption (FPE) and date-shifting.

In [ ]:
import os
import hashlib
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

# Set required salt for FPE
os.environ['EHDIP_HASH_SALT'] = 'super_secret_salt_12345'
os.environ['SPARK_VERSION'] = '3.3'

spark = SparkSession.builder \
    .appName("Day04_DeID_Engine") \
    .getOrCreate()

print("SparkSession initialized.")

In [ ]:
# Replicate deid_engine logic for the notebook demo

def get_hash_salt():
    salt = os.environ.get('EHDIP_HASH_SALT')
    if not salt:
        raise ValueError("Environment variable EHDIP_HASH_SALT must be set for FPE.")
    return salt

def fpe_mask(value: str, salt: str) -> str:
    if not value:
        return value
    salted = value + salt
    return hashlib.sha256(salted.encode('utf-8')).hexdigest()[:12] # Truncated for display

fpe_udf = F.udf(lambda x: fpe_mask(x, get_hash_salt()) if x else x, StringType())

def date_shift(patient_id_col: F.Column, date_col: F.Column) -> F.Column:
    # Deterministic shift between -30 and +30 days
    shift_days = F.pmod(F.abs(F.hash(patient_id_col)), F.lit(61)) - F.lit(30)
    return F.expr(f"date_add({date_col._jc.toString()}, {shift_days._jc.toString()})")


In [ ]:
# Generate synthetic raw patient data
data = [
    ("P001", "123-45-6789", "MRN-1001", "2023-01-15", "Patient feels fine. Call at 555-1234 or SSN 123-45-6789."),
    ("P002", "987-65-4321", "MRN-2002", "2023-05-20", "No complaints today."),
    ("P003", "111-22-3333", "MRN-3003", "2023-11-05", "Refill prescription. SSN on file: 111-22-3333.")
]
columns = ["patient_id", "ssn", "mrn", "encounter_date", "clinical_notes"]

df_raw = spark.createDataFrame(data, columns)
df_raw = df_raw.withColumn("encounter_date", F.to_date(F.col("encounter_date")))

print("Raw Data:")
df_raw.show(truncate=False)

In [ ]:
# Apply De-Identification
df_masked = df_raw \
    .withColumn("ssn_masked", fpe_udf(F.col("ssn"))) \
    .withColumn("mrn_masked", fpe_udf(F.col("mrn"))) \
    .withColumn("encounter_date_shifted", date_shift(F.col("patient_id"), F.col("encounter_date"))) \
    .withColumn("clinical_notes_redacted", F.regexp_replace(F.col("clinical_notes"), r'\b\d{3}-\d{2}-\d{4}\b', '[REDACTED_SSN]'))

print("Masked Data (De-Identified):")
df_masked.select("patient_id", "ssn_masked", "mrn_masked", "encounter_date_shifted", "clinical_notes_redacted").show(truncate=False)